In [54]:
import torch    
import torch.nn.functional as F

In [ ]:
torch.manual_seed(1337)

def attention(q, k, v):
    score = q @ k.transpose(-2, -1) # (h, T, d_k) @ (h, d_k, T) -> (h, T, T) 
    score = score / q.shape[-1]**0.5 # shape (h, T, T)
    weights = F.softmax(score, dim=-1) #shape: (h, T, T)
    out = weights @ v # shape: (h, T, T) @ (h, T, d_k) -> (h, T, d_k)
    return out, weights

def multi_head_attention(emb, w_q, w_k, w_v, w_o, n_heads):
    T, dim = emb.shape
    d_k = dim // n_heads

    q = emb @ w_q # (T, dim) @ (dim, dim) -> (T, dim)
    k = emb @ w_k # (T, dim) @ (dim, dim) -> (T, dim)
    v = emb @ w_v # (T, dim) @ (dim, dim) -> (T, dim)

    # split into heads: (T, dim) -> (T, n_heads, d_k) -> (n_heads, T, d_k)
    # note that n_heads * d_k -> dim
    q = q.view(T, n_heads, d_k).transpose(0, 1) # (n_heads, T, d_k)
    k = k.view(T, n_heads, d_k).transpose(0, 1) # (n_heads, T, d_k)
    v = v.view(T, n_heads, d_k).transpose(0, 1) # (n_heads, T, d_k)

    out, weights = attention(q, k, v)

    # merge heads back: (n_heads, T, d_k) -> (T, n_heads, d_k) -> (T, dim)
    out = out.transpose(0, 1).contiguous().view(T, dim)

    final = out @ w_o # (T, dim) @ (dim, dim) => (T, dim)
    return final, weights

def layer_norm(x, gamma, beta,  ):
    mean = x.mean(dim=1, keepdim=True) # get mean across rows - get the mean of each token's representation
    std = x.std(dim=1, keepdim=True) # get std across rows - get the std of each token's representation 
    x_norm = (x - mean) / (std + eps) # layer norm 
    return gamma * x_norm + beta 

def positional_encoding(seq_len, dim):
    pos = torch.arange(seq_len).unsqueeze(1) # shape: (seq_len, 1)
    i = torch.arange(0, dim, 2) # shape: (dim/2,)
    freq = 1.0 / (10000 ** (i / dim)) # shape: (dim/2,)
    angles = pos * freq # shape: (seq_len, dim/2)
    pe = torch.zeros(seq_len, dim) # shape: (seq_len, dim)
    pe[:, 0::2] = torch.sin(angles) # apply sin to even indices
    pe[:, 1::2] = torch.cos(angles) # apply cos to odd indices
    return pe

n_heads = 4
dim = 16
emb = torch.randn(3, dim) # shape (3, 16)

w_q = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_k = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_v = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)
w_o = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim)

gamma = torch.ones(dim)
beta = torch.zeros(dim)

attn_out, weights = multi_head_attention(emb, w_q, w_k, w_v, w_o, n_heads)
x = layer_norm(emb + attn_out, gamma, beta) # residual connection + layer norm

# Parameters for the feed forward network
ffn_w1 = torch.randn(dim, dim * 4) / (dim**0.5) # shape: (dim, dim * 4)
ffn_b1 = torch.randn(dim * 4) / (dim**0.5) # shape: (dim * 4)
ffn_w2 = torch.randn(dim * 4, dim) / (dim**0.5) # shape: (dim * 4, dim)
ffn_b2 = torch.randn(dim) / (dim**0.5) # shape: (dim)

def feed_forward(x, w1, b1, w2, b2):
    x = F.relu(x @ w1 + b1) # shape: (T, dim * 4)
    x = x @ w2 + b2 # shape: (T, dim)
    return x

ffn_out = feed_forward(x, ffn_w1, ffn_b1, ffn_w2, ffn_b2)

gamma1 = torch.ones(dim)
beta1 = torch.zeros(dim)

ffn_out = layer_norm(x + ffn_out, gamma1, beta1) # residual connection + layer norm


def encoder_block(emb, w_q, w_k, w_v, w_o, n_heads, gamma, beta, ffn_w1, ffn_b1, ffn_w2, ffn_b2):
    attn_out, weights = multi_head_attention(emb, w_q, w_k, w_v, w_o, n_heads)
    x = layer_norm(emb + attn_out, gamma, beta) # residual connection + layer norm
    ffn_out = feed_forward(x, ffn_w1, ffn_b1, ffn_w2, ffn_b2)
    gamma1 = torch.ones(x.shape[1])
    beta1 = torch.zeros(x.shape[1])
    ffn_out = layer_norm(x + ffn_out, gamma1, beta1) # residual connection + layer norm
    return ffn_out

# block 1 
x1 = encoder_block(emb, w_q, w_k, w_v, w_o, n_heads, gamma, beta, ffn_w1, ffn_b1, ffn_w2, ffn_b2)
#X1.shape


# Block 2 - weights, freshly initialized, same shapes as before
w_q2 = torch.randn(dim, dim) / dim**0.5
w_k2 = torch.randn(dim, dim) / dim**0.5
w_v2 = torch.randn(dim, dim) / dim**0.5
w_o2 = torch.randn(dim, dim) / dim**0.5

gamma_2 = torch.ones(dim)
beta_2 = torch.zeros(dim)

# Parameters for the feed forward network
ffn_w2_1 = torch.randn(dim, dim * 4) / (dim**0.5) # shape: (dim, dim * 4)
ffn_b2_1 = torch.randn(dim * 4) / (dim**0.5) # shape: (dim * 4)
ffn_w2_2 = torch.randn(dim * 4, dim) / (dim**0.5) # shape: (dim * 4, dim)
ffn_b2_2 = torch.randn(dim) / (dim**0.5) # shape: (dim)

#x2 = encoder_block(x1, w_q2, w_k2, w_v2, w_o2, n_heads, gamma_2, beta_2, ffn_w2_1, ffn_b2_1, ffn_w2_2, ffn_b2_2)

#x2.shape


torch.Size([3, 16])

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(1337)


torch.manual_seed(0)

vocab_size = 10
seq_len = 4
dim = 16

# -----------------------Build Datasets----------------------

def make_example(vocab_size, seq_len):
    x = torch.randint(0, vocab_size, (seq_len,))
    y = x.flip(0)
    
    return x, y

x, y = make_example(vocab_size, seq_len)
print(x.shape)

# ----------------------Positional Encoding----------------------
def positional_encoding(seq_len, dim):
    pos = torch.arange(seq_len).unsqueeze(1) # shape: (seq_len, 1)
    i = torch.arange(0, dim, 2) # shape: (dim/2,)
    freq = 1.0 / (10000 ** (i / dim)) # shape: (dim/2,)
    angles = pos * freq # shape: (seq_len, dim/2)
    pe = torch.zeros(seq_len, dim) # shape: (seq_len, dim)
    pe[:, 0::2] = torch.sin(angles) # apply sin to even indices
    pe[:, 1::2] = torch.cos(angles) # apply cos to odd indices
    return pe


# ----------------------Attention----------------------

def attention(q, k, v):
    score = q @ k.transpose(-2, -1) # (h, vocab_size, d_k) @ (h, d_k, vocab_size) -> (h, vocab_size, vocab_size) 
    score = score / q.shape[-1]**0.5 # shape (h, vocab_size, vocab_size)
    weights = F.softmax(score, dim=-1) #shape: (h, vocab_size, vocab_size)
    out = weights @ v # shape: (h, vocab_size, vocab_size) @ (h, vocab_size, d_k) -> (h, vocab_size, d_k)
    return out, weights


def multi_head_attention(emb_pe, w_q, w_k, w_v, w_o, n_heads):
    vocab_size, dim = emb_pe.shape
    d_k = dim // n_heads

    q = emb_pe @ w_q # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)
    k = emb_pe @ w_k # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)
    v = emb_pe @ w_v # (vocab_size, dim) @ (dim, dim) -> (vocab_size, dim)

    # split into heads: (vocab_size, dim) -> (vocab_size, h, d_k) -> (h, vocab_size, d_k)
    # note that h * d_k -> dim
    q = q.view(vocab_size, n_heads, d_k).transpose(0, 1) 
    k = k.view(vocab_size, n_heads, d_k).transpose(0, 1)
    v = v.view(vocab_size, n_heads, d_k).transpose(0, 1)

    out, weights = attention(q, k, v)

    # merge heads back: (h, vocab_size, d_k) -> (vocab_size, h, d_k) -> (vocab_size, dim)
    out = out.transpose(0, 1).contiguous().view(vocab_size, dim)

    final = out @ w_o
    return final, weights

# ----------------------Layer Normalization----------------------
def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=1, keepdim=True) # get mean across rows - get the mean of each token's representation
    std = x.std(dim=1, keepdim=True) # get std across rows - get the std of each token's representation 
    x_norm = (x - mean) / (std + eps) # layer norm 
    return gamma * x_norm + beta 

#-----------------------Feed Forward Network----------------------

def feed_forward(x, w1, b1, w2, b2):
    x = F.relu(x @ w1 + b1) # shape: (vocab_size, dim * 4) # (4, 64)
    x = x @ w2 + b2 # shape: (vocab_size, dim) # (4, 16)
    return x


#----------------------Encoder Block------------------------------------

# Weight matrices for multi-head attention
w_q = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim) -> (16, 16)
w_k = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim) -> (16, 16)
w_v = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim) -> (16, 16)
w_o = torch.randn(dim, dim) / dim**0.5 # shape: (dim, dim) -> (16, 16)

# Layer normalization parameters
gamma1 = torch.ones(dim) # shape: (dim) -> (16,)
beta1 = torch.zeros(dim) # shape: (dim) -> (16,)
gamma2 = torch.ones(dim) # shape: (dim) -> (16,)
beta2 = torch.zeros(dim) # shape: (dim) -> (16,)

# Parameters for the feed forward network
ffn_w1 = torch.randn(dim, dim * 4) / (dim**0.5) # shape: (dim, dim * 4) # (16, 64)
ffn_b1 = torch.randn(dim * 4) / (dim**0.5) # shape: (dim * 4) # (64,)
ffn_w2 = torch.randn(dim * 4, dim) / (dim**0.5) # shape: (dim * 4, dim) # (64, 16)
ffn_b2 = torch.randn(dim) / (dim**0.5) # shape: (dim)  # (16,)

def encoder_block(emb_pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2):
    attn_out, weights = multi_head_attention(emb_pe, w_q, w_k, w_v, w_o, n_heads)
    x = layer_norm(emb_pe + attn_out, gamma1, beta1) # residual connection + layer norm
    ffn_out = feed_forward(x, ffn_w1, ffn_b1, ffn_w2, ffn_b2)
    ffn_out = layer_norm(x + ffn_out, gamma2, beta2) # residual connection + layer norm
    return ffn_out


# -------------------- Forward Function ----------------------------------------------

# attention heads
n_heads = 4

# Embedding matrix
C = torch.randn(vocab_size, dim) / (dim**0.5) # shape: (vocab_size, dim) => (10, 16)

# output projection parameters
w_out = torch.randn(dim, vocab_size) * 0.01 # shape: (dim, vocab_size) -> (16, 10)
b_out = torch.randn(vocab_size) * 0 # shape: (vocab_size) -> (10,)

pe = positional_encoding(seq_len, dim) # shape: (seq_len, dim) (4, 16)

def forward(x, C, pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2, w_out, b_out):
    emb = C[x] # shape: (seq_len, dim) (4, 16)
    emb_pe = emb + pe # shape: (seq_len, dim) (4, 16)
    encoder_out = encoder_block(emb_pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2)
    logits = encoder_out @ w_out + b_out # shape: (seq_len, vocab_size) (4, 10)
    return logits


logits = forward(x, C, pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2, w_out, b_out) # shape: (seq_len, vocab_size) (4, 10)
loss = F.cross_entropy(logits, y) 
loss 



torch.Size([4])


tensor(2.3249)

In [202]:
params = [C, w_q, w_k, w_v, w_o, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2, w_out, b_out]

for p in params:
    p.requires_grad = True

In [203]:
lr = 0.01

for step in range(10000):
    # load dataset 
    x, y = make_example(vocab_size, seq_len)

    # forward pass
    logits = forward(x, C, pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2, w_out, b_out)
    loss = F.cross_entropy(logits, y)

    for p in params:
        if p.grad is not None:
            p.grad = None # clear gradients
    
    # Backward pass
    loss.backward()
    
    # Update 
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    if step % 2000 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

Step 0, Loss: 2.273677110671997
Step 2000, Loss: 1.5839803218841553
Step 4000, Loss: 0.3581755757331848
Step 6000, Loss: 0.020444322377443314
Step 8000, Loss: 0.010927317664027214


In [204]:
x, y = make_example(vocab_size, seq_len)
logits = forward(x, C, pe, w_q, w_k, w_v, w_o, n_heads, gamma1, beta1, gamma2, beta2, ffn_w1, ffn_b1, ffn_w2, ffn_b2, w_out, b_out)
pred = logits.argmax(dim=-1)

print("input:   ", x)
print("target:  ", y)
print("predicted:", pred)

input:    tensor([0, 7, 3, 6])
target:   tensor([6, 3, 7, 0])
predicted: tensor([6, 3, 7, 0])


In [168]:
x

tensor([4, 9, 3, 0])

In [121]:
vocab_size = 10 
dim = 16
C = torch.randn(vocab_size, dim) / (dim**0.5) # shape: (vocab_size, dim)

tokens = torch.tensor([3, 1, 4, 2]) # shape: (seq_len,)
emb = C[tokens] # shape: (seq_len, dim)

emb

tensor([[ 0.0090, -0.0856, -0.0954, -0.0142, -0.2192,  0.1306,  0.1155, -0.3323,
          0.1277, -0.0715,  0.0838,  0.2930, -0.1010, -0.0816,  0.1871, -0.0488],
        [-0.0411, -0.2429, -0.2577,  0.1618, -0.3646, -0.0704, -0.2575, -0.1721,
          0.0549, -0.4247,  0.3274, -0.4153,  0.4336,  0.1667, -0.2603, -0.3505],
        [-0.1718, -0.5837,  0.0235, -0.0505,  0.7814,  0.0446, -0.1613,  0.0028,
          0.1338,  0.4933, -0.0519, -0.0076,  0.0649,  0.1529,  0.2954, -0.2762],
        [ 0.0525,  0.0075,  0.4273, -0.1814, -0.3633, -0.0119,  0.3749,  0.1358,
          0.1848,  0.3133, -0.1111,  0.2046,  0.0031,  0.2439, -0.0020, -0.0100]])

In [73]:
pe = positional_encoding(3, 16)
pe.shape

torch.Size([3, 16])